# 02 — Quy trình xử lý dữ liệu

**Nguyên tắc chống rò rỉ:** chia train/test TRƯỚC, `fit` scaler/encoder **chỉ trên train**, `transform` test. Toàn bộ gói trong `sklearn Pipeline` / `ColumnTransformer`.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..") / "src"))
from preprocess import (
    load_raw_dataframe, inspect_and_clean, split_xy, build_preprocessor,
    FEATURE_ORDER, NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET,
)

df = inspect_and_clean(load_raw_dataframe())
X_train, X_test, y_train, y_test = split_xy(df)
print("train", X_train.shape, "test", X_test.shape)
print("features", FEATURE_ORDER)
print("target", TARGET)
pre = build_preprocessor()
pre


## Vì sao từng bước?

| Bước | Làm gì | Vì sao |
|---|---|---|
| Kiểm tra | dtype, duplicate, missing, min/max | Dữ liệu Kaggle sạch; 1 duplicate giữ lại |
| Làm sạch | không drop outlier smoker | chi phí cao là tín hiệu, không phải nhiễu |
| Imputer | median / most_frequent | dataset không thiếu, nhưng API có thể nhận null |
| One-Hot | sex, smoker, region | danh mục **không thứ tự** → không dùng Ordinal |
| StandardScaler | age, bmi, children | SVR và Linear cần; cây không hại |
| Split 80/20, seed=42 | hồi quy, không stratify | test 268 dòng đủ ổn định metric |

Pipeline này được **lưu cùng model** (`model.joblib`), AI Service không viết lại preprocessor.


In [ ]:
# Minh họa: fit CHỈ trên train
Xt = pre.fit_transform(X_train)
print("train transformed", Xt.shape)
print("test transformed", pre.transform(X_test).shape)
print("không fit trên test — tránh leakage")
